<a href="https://colab.research.google.com/github/Sarah-0405/Cold_Spots_Bayern/blob/main/KNN_Gi_Cold_Spot_Berechnungen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Dieses Notebook berechnet eine KNN-Matrix sowie Gi*-Klassifizierung in Cold Spots, Hot Spots, Neutral Spots für
- jede Stadt über 50.000 EW der Jahre 2019-2024 einzeln (funktioniert bisher noch nicht, zu wenig RAM-Speicher)
- **jede Stadt über 50.000 EW im Gesamtzeitraum 2019-2024** (ab Kapitel "KNN für Mittelwert aller Sommer")
    - dafür wird zuerst ein gemeinsamer gdf erstellt, der für jeden Pixel den Durchschnitts-LST-Wert aller Sommerszenen von 2019-2024 speichert (in drive: averaged_lst_per_pixel_allyears.geojson)
    - daraus werden dann lokale gdf pro Stadt erstellt und auf Basis dessen für jede Stadt eine KNN-Matrix (drive Ordner: Cold Spots Bayern > knn_weights_averaged, jede Stadt einzeln gespeichert)
    - für **Gi*-Analyse** wird jede Stadt in mehreren Chunks verarbeitet, dafür werden Gitterzellen erstellt, die dann nacheinander verarbeitet werden
      - die Gitterzellen werden mit den lokalen gdfs der LST-Mittelwerte jeder Stadt gejoined und Gi* Analyse durchgeführt Drive ordner: Cold Spots Bayern > gi_results_chunked (Ergebnisse der einzelnen chunks vorher wieder zusammengefügt zu gesamtem Stadtgebiet))

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install geemap
!pip install geopandas
!pip install geopy
!pip install folium
!pip install matplotlib
!pip install numpy
!pip install pandas
!pip install rasterio
!pip install seaborn
!pip install shapely
!pip install sklearn
!pip install pysal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 35.5 MB/s eta 0:00:00
  error: subprocess-exited-with-error
  
  × python setup.py egg_info did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Preparing metadata (setup.py) ... error
error: metadata-generation-failed

× Encountered error while generating package metadata.
╰─> See above for output.

note: This is an issue with the package mentioned above, not pip.
hint: See above for details.
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.2/882.2 kB 4.8 MB/s eta 0:00:00
 

In [2]:
!pip install pysal

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.6/56.6 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 882.2/882.2 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.9/47.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.2/59.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.6/141.6 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.1/248.1 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 26.7 MB/s eta 0:00:00
   ━━━━━

In [2]:
# Importieren Sie notwendige Bibliotheken
import ee
import geemap
import folium
from datetime import datetime
import geopandas as gpd
from shapely.geometry import mapping, Point
import pandas as pd
import matplotlib.pyplot as plt
import pysal.lib as ps
import pysal.explore as pe
from esda.getisord import G_Local
from libpysal.weights import KNN
from sklearn.preprocessing import StandardScaler
import numpy as np
import os

/usr/local/lib/python3.11/dist-packages/spaghetti/network.py:41: FutureWarning: The next major release of pysal/spaghetti (2.0.0) will drop support for all ``libpysal.cg`` geometries. This change is a first step in refactoring ``spaghetti`` that is expected to result in dramatically reduced runtimes for network instantiation and operations. Users currently requiring network and point pattern input as ``libpysal.cg`` geometries should prepare for this simply by converting to ``shapely`` geometries.
  warnings.warn(dep_msg, FutureWarning, stacklevel=1)


# KNN für Mittelwert aller Sommer

**KNN für Durchschnitt der Jahre 2019-2025** erstellen => erstmal gdfs Mittelwert der Jahre, dann KNN

In [ ]:
all_cities_summer_avg_lst_per_pixel = gpd.read_file("/content/drive/MyDrive/Cold Spots Bayern/all_cities_summer_avg_lst_per_pixel_allyears.geojson")
display(all_cities_summer_avg_lst_per_pixel.head())

In [ ]:
# 1. Gruppierung und Mittelwertbildung
# Gruppieren nach 'city' und 'geometry' und berechnen des Mittelwerts für 'avg_summer_LST_Celsius'
# Setzen Sie 'geometry' als Index, um die Gruppierung zu vereinfachen und dann zurückzusetzen
averaged_lst_per_pixel_per_city = all_cities_summer_avg_lst_per_pixel.set_index('geometry').groupby(['city', 'geometry'])['avg_summer_LST_Celsius'].mean().reset_index()

# Konvertieren Sie das Ergebnis zurück in ein GeoDataFrame
# Annahme: Das ursprüngliche CRS des GeoDataFrames ist bekannt oder kann von der ersten Zeile abgeleitet werden.
# Wenn das ursprüngliche CRS nicht bekannt ist, müssen Sie es hier explizit festlegen.
# Beispiel: original_crs = all_cities_summer_avg_lst_per_pixel.crs
# Wenn das CRS None ist und Sie wissen, dass es sich um WGS84 (EPSG:4326) handelt:
# original_crs = "EPSG:4326"

# Verwenden Sie das CRS des ursprünglichen GeoDataFrames
original_crs = all_cities_summer_avg_lst_per_pixel.crs

averaged_lst_per_pixel_per_city = gpd.GeoDataFrame(
    averaged_lst_per_pixel_per_city,
    geometry='geometry',
    crs=original_crs # Setzen Sie das CRS des ursprünglichen GeoDataFrames
)

print("Mittelwert der LST pro Pixel pro Stadt über alle Jahre berechnet.")
display(averaged_lst_per_pixel_per_city.head())

Mittelwert der LST pro Pixel pro Stadt über alle Jahre berechnet.


,city,geometry,avg_summer_LST_Celsius
0,Aschaffenburg,POINT (9.23257 49.93527),25.827932
1,Aschaffenburg,POINT (9.2355 49.935),25.828958
2,Aschaffenburg,POINT (9.2355 49.93527),25.846390
3,Aschaffenburg,POINT (9.23591 49.935),25.713771
4,Aschaffenburg,POINT (9.23591 49.93527),25.751711


## Geodataframe 2019-2024 Durchschnitt speichern

In [ ]:
# Definieren Sie den Pfad zum Speichern der Datei in Google Drive
# Passen Sie den Ordner und Dateinamen bei Bedarf an
output_path_averaged_gdf = "/content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson"

try:
    # Speichern Sie den GeoDataFrame als GeoJSON-Datei
    averaged_lst_per_pixel_per_city.to_file(output_path_averaged_gdf, driver='GeoJSON')
    print(f"GeoDataFrame mit gemittelten LST-Werten erfolgreich gespeichert unter: {output_path_averaged_gdf}")
except Exception as e:
    print(f"Fehler beim Speichern des GeoDataFrames: {e}")

GeoDataFrame mit gemittelten LST-Werten erfolgreich gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson


gespeicherten gdf laden und damit weiterarbeiten: enthält die Mittelwerte aller verfügbaren Sommerszenen 2019-2024

In [3]:
# Definieren Sie den Pfad, von dem die Datei geladen werden soll
# Stellen Sie sicher, dass dies mit dem Pfad übereinstimmt, unter dem Sie die Datei gespeichert haben
input_path_averaged_gdf = "/content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson"

try:
    # Laden Sie den GeoDataFrame aus der GeoJSON-Datei
    loaded_averaged_lst_gdf = gpd.read_file(input_path_averaged_gdf)
    print(f"GeoDataFrame erfolgreich geladen von: {input_path_averaged_gdf}")
    print(f"Anzahl der Einträge im geladenen GeoDataFrame: {len(loaded_averaged_lst_gdf)}")
    display(loaded_averaged_lst_gdf.head())

except Exception as e:
    print(f"Fehler beim Laden des GeoDataFrames von {input_path_averaged_gdf}: {e}")
    print("Bitte überprüfen Sie den Dateipfad und stellen Sie sicher, dass die Datei existiert.")

# Nun können Sie mit 'loaded_averaged_lst_gdf' weiterarbeiten
# Zum Beispiel können Sie es in city_averaged_gdfs aufteilen oder direkt verwenden
# loaded_averaged_lst_gdf sollte die gleiche Struktur wie 'averaged_lst_per_pixel_per_city' haben

GeoDataFrame erfolgreich geladen von: /content/drive/MyDrive/Cold Spots Bayern/averaged_lst_per_pixel_per_city_allyears.geojson
Anzahl der Einträge im geladenen GeoDataFrame: 1992069


,city,avg_summer_LST_Celsius,geometry
0,Aschaffenburg,25.827932,POINT (9.23257 49.93527)
1,Aschaffenburg,25.828958,POINT (9.2355 49.935)
2,Aschaffenburg,25.846390,POINT (9.2355 49.93527)
3,Aschaffenburg,25.713771,POINT (9.23591 49.935)
4,Aschaffenburg,25.751711,POINT (9.23591 49.93527)


In [4]:
# 2. Erstellung stadtspezifischer GeoDataFrames
# Dictionary zum Speichern der stadtspezifischen GeoDataFrames mit gemittelten Werten
city_averaged_gdfs = {}

unique_cities_averaged = loaded_averaged_lst_gdf['city'].unique()
print(f"\nEindeutige Städte im gemittelten GeoDataFrame: {list(unique_cities_averaged)}")


print("\nErstelle stadtspezifische GeoDataFrames mit gemittelten LST-Werten...")

for city in unique_cities_averaged:
    print(f"  Erstelle GeoDataFrame für Stadt: {city}...")
    gdf_city_averaged = loaded_averaged_lst_gdf[
        loaded_averaged_lst_gdf['city'] == city
    ].copy() # Wichtig: Kopie erstellen

    if not gdf_city_averaged.empty:
        city_averaged_gdfs[city] = gdf_city_averaged
        print(f"    GeoDataFrame für {city} erstellt mit {len(gdf_city_averaged)} Pixeln.")
    else:
        print(f"    Keine gemittelten Daten für Stadt {city} gefunden.")

print("\nStadtspezifische GeoDataFrames mit gemittelten LST-Werten erstellt.")

# Jetzt haben Sie ein Dictionary 'city_averaged_gdfs', das für jede Stadt einen GeoDataFrame
# mit den gemittelten LST-Werten pro Pixel über alle Jahre enthält.
# Beispiel: city_averaged_gdfs['Munich'] gibt den GeoDataFrame für München mit gemittelten LST-Werten zurück.


Eindeutige Städte im gemittelten GeoDataFrame: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']

Erstelle stadtspezifische GeoDataFrames mit gemittelten LST-Werten...
  Erstelle GeoDataFrame für Stadt: Aschaffenburg...
    GeoDataFrame für Aschaffenburg erstellt mit 69391 Pixeln.
  Erstelle GeoDataFrame für Stadt: Augsburg...
    GeoDataFrame für Augsburg erstellt mit 164425 Pixeln.
  Erstelle GeoDataFrame für Stadt: Bamberg...
    GeoDataFrame für Bamberg erstellt mit 121604 Pixeln.
  Erstelle GeoDataFrame für Stadt: Bayreuth...
    GeoDataFrame für Bayreuth erstellt mit 149925 Pixeln.
  Erstelle GeoDataFrame für Stadt: Erlangen...
    GeoDataFrame für Erlangen erstellt mit 80705 Pixeln.
  Erstelle GeoDataFrame für Stadt: Fürth...
    GeoDataFrame für Fürth erstellt mit 60360 Pixeln.
  Erstelle GeoDataFrame für Stadt: Ingolstadt

In [ ]:
# 3. KNN-Matrix Berechnung pro Stadt
# Dictionary zum Speichern der KNN-Gewichtsmatrizen für jede Stadt
city_knn_weights_averaged = {}

# Definieren Sie die Anzahl der Nachbarn für KNN
k_neighbors = 12 # Sie können diesen Wert anpassen

print(f"\nBerechne KNN-Gewichtsmatrizen (k={k_neighbors}) für jede Stadt basierend auf gemittelten LST-Werten...")

# Schleife über die stadtspezifischen GeoDataFrames
for city, gdf_city_averaged in city_averaged_gdfs.items():
    print(f"\n✨ Verarbeite Stadt: {city}...")

    # Stellen Sie sicher, dass genügend Punkte für die KNN-Analyse vorhanden sind
    min_points_for_knn = k_neighbors + 1
    if not gdf_city_averaged.empty and len(gdf_city_averaged) >= min_points_for_knn:
        # Extrahieren Sie die Koordinaten
        # Konvertiere in ein metrisches CRS vor der KNN-Berechnung, wenn es noch nicht projiziert ist
        gdf_city_spatial = gdf_city_averaged.copy() # Kopie erstellen
        if gdf_city_spatial.crs is None or gdf_city_spatial.crs.is_geographic:
             # Für Bayern ist EPSG:25832 (ETRS89 / UTM zone 32N) oft passend.
             try:
                 print(f"    Konvertiere Daten für {city} nach EPSG:25832 (UTM 32N) für metrische Distanzen.")
                 gdf_city_spatial = gdf_city_spatial.to_crs(epsg=25832)
             except Exception as crs_e:
                 print(f"    Fehler bei der CRS-Konvertierung für {city}: {crs_e}. Versuche es ohne Konvertierung, Ergebnisse könnten ungenau sein.")
                 pass # Geht weiter mit den ursprünglichen Koordinaten

        coords = np.array(list(zip(gdf_city_spatial.geometry.x, gdf_city_spatial.geometry.y)))

        # Berechne die KNN-Matrix
        try:
            w_city_averaged = KNN.from_array(coords, k=k_neighbors)
            w_city_averaged.transform = 'R' # Zeilenstandardisierung anwenden

            # Prüfen auf Konnektivität
            if w_city_averaged.n_components > 1:
                 print(f"    Warnung: Gewichtsmatrix für {city} ist nicht vollständig verbunden ({w_city_averaged.n_components} Komponenten).")

            # Speichere die Gewichtsmatrix im Dictionary
            city_knn_weights_averaged[city] = w_city_averaged
            print(f"    KNN-Matrix für {city} erfolgreich berechnet.")

        except Exception as knn_e:
            print(f"    Fehler bei der KNN-Berechnung für {city}: {knn_e}. Überspringe diese Stadt.")


    else:
        print(f"  Nicht genügend Daten ({len(gdf_city_averaged)} Punkte) für KNN-Berechnung für {city}. Benötigt mindestens {min_points_for_knn} Punkte. Überspringe.")


print("\nKNN-Berechnung für alle Städte (basierend auf gemittelten LST-Werten) abgeschlossen.")

# Das Dictionary 'city_knn_weights_averaged' enthält nun die KNN-Matrizen
# für jede Stadt, basierend auf den über die Jahre gemittelten LST-Werten pro Pixel.
# Beispiel: city_knn_weights_averaged['Munich'] gibt die KNN-Matrix für München zurück.


Berechne KNN-Gewichtsmatrizen (k=12) für jede Stadt basierend auf gemittelten LST-Werten...

✨ Verarbeite Stadt: Aschaffenburg...
    Konvertiere Daten für Aschaffenburg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Aschaffenburg erfolgreich berechnet.

✨ Verarbeite Stadt: Augsburg...
    Konvertiere Daten für Augsburg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Augsburg erfolgreich berechnet.

✨ Verarbeite Stadt: Bamberg...
    Konvertiere Daten für Bamberg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Bamberg erfolgreich berechnet.

✨ Verarbeite Stadt: Bayreuth...
    Konvertiere Daten für Bayreuth nach EPSG:25832 (UTM 32N) für metrische Distanzen.


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


    Warnung: Gewichtsmatrix für Bayreuth ist nicht vollständig verbunden (2 Komponenten).
    KNN-Matrix für Bayreuth erfolgreich berechnet.

✨ Verarbeite Stadt: Erlangen...
    Konvertiere Daten für Erlangen nach EPSG:25832 (UTM 32N) für metrische Distanzen.


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


    Warnung: Gewichtsmatrix für Erlangen ist nicht vollständig verbunden (10 Komponenten).
    KNN-Matrix für Erlangen erfolgreich berechnet.

✨ Verarbeite Stadt: Fürth...
    Konvertiere Daten für Fürth nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Fürth erfolgreich berechnet.

✨ Verarbeite Stadt: Ingolstadt...
    Konvertiere Daten für Ingolstadt nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Ingolstadt erfolgreich berechnet.

✨ Verarbeite Stadt: Kempten (Allgäu)...
    Konvertiere Daten für Kempten (Allgäu) nach EPSG:25832 (UTM 32N) für metrische Distanzen.


...

    Warnung: Gewichtsmatrix für Kempten (Allgäu) ist nicht vollständig verbunden (3 Komponenten).
    KNN-Matrix für Kempten (Allgäu) erfolgreich berechnet.

✨ Verarbeite Stadt: Landshut...
    Konvertiere Daten für Landshut nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Landshut erfolgreich berechnet.

✨ Verarbeite Stadt: Munich...
    Konvertiere Daten für Munich nach EPSG:25832 (UTM 32N) für metrische Distanzen.


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


    Warnung: Gewichtsmatrix für Munich ist nicht vollständig verbunden (3 Komponenten).
    KNN-Matrix für Munich erfolgreich berechnet.

✨ Verarbeite Stadt: Nuremberg...
    Konvertiere Daten für Nuremberg nach EPSG:25832 (UTM 32N) für metrische Distanzen.


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


    Warnung: Gewichtsmatrix für Nuremberg ist nicht vollständig verbunden (2 Komponenten).
    KNN-Matrix für Nuremberg erfolgreich berechnet.

✨ Verarbeite Stadt: Passau...
    Konvertiere Daten für Passau nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Passau erfolgreich berechnet.

✨ Verarbeite Stadt: Regensburg...
    Konvertiere Daten für Regensburg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Regensburg erfolgreich berechnet.

✨ Verarbeite Stadt: Rosenheim...
    Konvertiere Daten für Rosenheim nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Rosenheim erfolgreich berechnet.

✨ Verarbeite Stadt: Schweinfurt...
    Konvertiere Daten für Schweinfurt nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Schweinfurt erfolgreich berechnet.

✨ Verarbeite Stadt: Würzburg...
    Konvertiere Daten für Würzburg nach EPSG:25832 (UTM 32N) für metrische Distanzen.
    KNN-Matrix für Würzburg erfolgreich berech

### KNN-Gewichtsmatrizen speichern

Dieser Code speichert jede KNN-Gewichtsmatrix aus dem `city_knn_weights_averaged` Dictionary als separate Datei.

In [ ]:
import os
import libpysal as ps # libpysal wird zum Speichern benötigt

# Definieren Sie den Ordner, in dem die Gewichtsmatrizen gespeichert werden sollen
# Passen Sie diesen Pfad bei Bedarf an, idealerweise in Ihrem Google Drive
output_weights_folder = "/content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/"
os.makedirs(output_weights_folder, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

print(f"Speichere KNN-Gewichtsmatrizen im Ordner: {output_weights_folder}")

# Schleife über das Dictionary und speichern Sie jede Gewichtsmatrix
for city, w_matrix in city_knn_weights_averaged.items():
    if w_matrix is not None:
        # Erstellen Sie einen Dateinamen basierend auf dem Stadtnamen
        # Verwenden Sie ein Format, das von libpysal unterstützt wird, z.B. .gal oder .gwt
        # Für KNN ist .gwt (General Weights format) oft passend, aber .gal funktioniert auch oft.
        # Wir verwenden hier .gal als Beispiel.
        file_name = f"{city.replace(' ', '_')}_knn_averaged.gal"
        file_path = os.path.join(output_weights_folder, file_name)

        try:
            # Speichern Sie die Gewichtsmatrix mit libpysal.io.open()
            # Verwenden Sie nicht den context manager (with), da GalIO ihn nicht unterstützt
            f = ps.io.open(file_path, mode='w')
            f.write(w_matrix)
            f.close() # Datei explizit schließen

            print(f"  Gewichtsmatrix für {city} gespeichert unter: {file_path}")
        except Exception as e:
            print(f"  Fehler beim Speichern der Gewichtsmatrix für {city}: {e}")
    else:
        print(f"  Keine Gewichtsmatrix für {city} verfügbar zum Speichern.")

print("\nSpeichern der KNN-Gewichtsmatrizen abgeschlossen.")

Speichere KNN-Gewichtsmatrizen im Ordner: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/
  Gewichtsmatrix für Aschaffenburg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Aschaffenburg_knn_averaged.gal
  Gewichtsmatrix für Augsburg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Augsburg_knn_averaged.gal
  Gewichtsmatrix für Bamberg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Bamberg_knn_averaged.gal
  Gewichtsmatrix für Bayreuth gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Bayreuth_knn_averaged.gal
  Gewichtsmatrix für Erlangen gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Erlangen_knn_averaged.gal
  Gewichtsmatrix für Fürth gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/Fürth_knn_averaged.gal
  Gewichtsmatrix für Ingolstadt gespeichert unter: /content/drive/MyDri

### KNN-Gewichtsmatrizen laden

Dieser Code lädt die zuvor gespeicherten KNN-Gewichtsmatrizen wieder in ein Dictionary.

In [ ]:
import os
import libpysal as ps # libpysal wird zum Laden benötigt

# Definieren Sie den Ordner, aus dem die Gewichtsmatrizen geladen werden sollen
input_weights_folder = "/content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/"

# Dictionary zum Speichern der geladenen Gewichtsmatrizen
loaded_city_knn_weights_averaged = {}

print(f"Lade KNN-Gewichtsmatrizen aus dem Ordner: {input_weights_folder}")

# Holen Sie sich eine Liste aller Gewichtsmatrizen-Dateien im Ordner
weight_files = [f for f in os.listdir(input_weights_folder) if f.endswith(".gal")] # Passen Sie die Dateiendung an, falls Sie ein anderes Format verwendet haben

# Schleife über die Dateien und laden Sie jede Gewichtsmatrix
for file_name in weight_files:
    file_path = os.path.join(input_weights_folder, file_name)
    # Extrahiere den Stadtnamen aus dem Dateinamen
    city_name = file_name.replace("_knn_averaged.gal", "").replace("_", " ") # Passen Sie dies an Ihren Dateinamen an

    try:
        # Laden Sie die Gewichtsmatrix
        w_matrix = ps.io.open(file_path, mode='r').read()
        # Wenn die Matrix zeilenstandardisiert war beim Speichern, bleibt sie das beim Laden.
        # Stellen Sie sicher, dass die Transformation korrekt ist, falls notwendig.
        # w_matrix.transform = 'R' # Optional: Zeilenstandardisierung erneut anwenden, falls nicht erhalten

        # Speichere die geladene Gewichtsmatrix im Dictionary
        loaded_city_knn_weights_averaged[city_name] = w_matrix
        print(f"  Gewichtsmatrix für {city_name} erfolgreich geladen.")
    except Exception as e:
        print(f"  Fehler beim Laden der Gewichtsmatrix aus {file_path}: {e}")

print("\nLaden der KNN-Gewichtsmatrizen abgeschlossen.")

# Das Dictionary 'loaded_city_knn_weights_averaged' enthält nun die geladenen Gewichtsmatrizen.
# Sie können dieses Dictionary anstelle von 'city_knn_weights_averaged' in Ihrer Gi* Analyse verwenden.

Lade KNN-Gewichtsmatrizen aus dem Ordner: /content/drive/MyDrive/Cold Spots Bayern/knn_weights_averaged/
  Gewichtsmatrix für Fürth erfolgreich geladen.
  Gewichtsmatrix für Bamberg erfolgreich geladen.


/usr/local/lib/python3.11/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Kempten (Allgäu) erfolgreich geladen.


/usr/local/lib/python3.11/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 10 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Erlangen erfolgreich geladen.
  Gewichtsmatrix für Aschaffenburg erfolgreich geladen.
  Gewichtsmatrix für Ingolstadt erfolgreich geladen.
  Gewichtsmatrix für Landshut erfolgreich geladen.


/usr/local/lib/python3.11/dist-packages/libpysal/io/iohandlers/gal.py:185: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  w = W(neighbors, id_order=ids)


  Gewichtsmatrix für Bayreuth erfolgreich geladen.
  Gewichtsmatrix für Augsburg erfolgreich geladen.
  Gewichtsmatrix für Munich erfolgreich geladen.
  Gewichtsmatrix für Nuremberg erfolgreich geladen.
  Gewichtsmatrix für Passau erfolgreich geladen.
  Gewichtsmatrix für Rosenheim erfolgreich geladen.
  Gewichtsmatrix für Schweinfurt erfolgreich geladen.
  Gewichtsmatrix für Würzburg erfolgreich geladen.


4. **Ergebnisse speichern/anzeigen**:

Die gemittelten GeoDataFrames pro Stadt sind im Dictionary `city_averaged_gdfs` gespeichert.
Die KNN-Gewichtsmatrizen pro Stadt sind im Dictionary `city_knn_weights_averaged` gespeichert.

Sie können diese Ergebnisse nun speichern oder weiter analysieren (z.B. Gi* Statistik berechnen).


# Gi* Analyse

In [ ]:
from esda.getisord import G_Local

Test: nur für Aschaffenburg

In [ ]:
def calculate_hotspots(gdf, w):
    gdf = gdf.copy()  # Kopie, um das Original nicht zu ändern
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Standardisierte Z-Werte
    return gdf

# Hol dir die Daten für Aschaffenburg
gdf_aschaffenburg = city_averaged_gdfs["Aschaffenburg"]
w_aschaffenburg = loaded_city_knn_weights_averaged["Aschaffenburg"]

# Berechne Hotspots
gdf_aschaffenburg_result = calculate_hotspots(gdf_aschaffenburg, w_aschaffenburg)

# Ergebnisse anschauen
print(gdf_aschaffenburg_result.head())


            city  avg_summer_LST_Celsius                  geometry       Gi*
0  Aschaffenburg               25.827932  POINT (9.23257 49.93527) -0.938972
1  Aschaffenburg               25.828958     POINT (9.2355 49.935) -1.077372
2  Aschaffenburg               25.846390   POINT (9.2355 49.93527) -1.221513
3  Aschaffenburg               25.713771    POINT (9.23591 49.935) -1.064959
4  Aschaffenburg               25.751711  POINT (9.23591 49.93527) -1.237013


In [ ]:
def calculate_hotspots(gdf, w):
    gdf = gdf.copy()  # Original nicht verändern
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs  # Standardisierte Z-Werte
    return gdf

# Ergebnisse für alle Städte berechnen
city_gi_results = {}

for city in city_averaged_gdfs.keys():
    print(f"✨ Berechne Gi* für: {city}")

    gdf = city_averaged_gdfs[city]
    w = loaded_city_knn_weights_averaged[city]

    # Berechne Hotspots
    result_gdf = calculate_hotspots(gdf, w)

    # Speichere das Ergebnis
    city_gi_results[city] = result_gdf

print("✅ Berechnung abgeschlossen.")

# Beispiel: Ergebnisse für Aschaffenburg ansehen
print(city_gi_results["Aschaffenburg"].head())


✨ Berechne Gi* für: Aschaffenburg
✨ Berechne Gi* für: Augsburg
✨ Berechne Gi* für: Bamberg
✨ Berechne Gi* für: Bayreuth
✨ Berechne Gi* für: Erlangen
✨ Berechne Gi* für: Fürth


KeyError: 'Fürth'

In [8]:
print("\n📋 Städte in city_averaged_gdfs:")
for c in sorted(city_averaged_gdfs.keys()):
    print(f"- {c}")

print("\n📋 Städte in loaded_city_knn_weights_averaged:")
for c in sorted(loaded_city_knn_weights_averaged.keys()):
    print(f"- {c}")



📋 Städte in city_averaged_gdfs:
- Aschaffenburg
- Augsburg
- Bamberg
- Bayreuth
- Erlangen
- Fürth
- Ingolstadt
- Kempten (Allgäu)
- Landshut
- Munich
- Nuremberg
- Passau
- Regensburg
- Rosenheim
- Schweinfurt
- Würzburg

📋 Städte in loaded_city_knn_weights_averaged:
- Aschaffenburg
- Augsburg
- Bamberg
- Bayreuth
- Erlangen
- Fürth
- Ingolstadt
- Kempten (Allgäu)
- Landshut
- Munich
- Nuremberg
- Passau
- Regensburg
- Rosenheim
- Schweinfurt
- Würzburg


In [ ]:
output_dir = "gi_star_results"
os.makedirs(output_dir, exist_ok=True)

def calculate_hotspots(gdf, w):
    gdf = gdf.copy()
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs
    return gdf

cities_gdfs = set(city_averaged_gdfs.keys())
cities_weights = set(loaded_city_knn_weights_averaged.keys())

print("\n📋 Städte in city_averaged_gdfs:", sorted(cities_gdfs))
print("📋 Städte in loaded_city_knn_weights_averaged:", sorted(cities_weights))

for city in sorted(cities_gdfs):
    if city not in loaded_city_knn_weights_averaged:
        print(f"⚠️  Kein Gewicht für {city} gefunden — überspringe.")
        continue

    print(f"\n✨ Bearbeite {city}...")

    gdf = city_averaged_gdfs[city]
    w = loaded_city_knn_weights_averaged[city]

    # Gi* berechnen
    result_gdf = calculate_hotspots(gdf, w)

    out_path = os.path.join(output_dir, f"{city.replace(' ', '_')}.geojson")

    result_gdf.to_file(out_path, driver="GeoJSON")
    print(f"✅ Ergebnis für {city} gespeichert: {out_path}")

    # Speicher freigeben
    del result_gdf



📋 Städte in city_averaged_gdfs: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']
📋 Städte in loaded_city_knn_weights_averaged: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']

✨ Bearbeite Aschaffenburg...
✅ Ergebnis für Aschaffenburg gespeichert: gi_star_results/Aschaffenburg.geojson

✨ Bearbeite Augsburg...
✅ Ergebnis für Augsburg gespeichert: gi_star_results/Augsburg.geojson

✨ Bearbeite Bamberg...
✅ Ergebnis für Bamberg gespeichert: gi_star_results/Bamberg.geojson

✨ Bearbeite Bayreuth...


start city jetzt immer ab da wo vorherige Sitzung abgestürzt ist

In [ ]:
output_dir = "gi_star_results"
os.makedirs(output_dir, exist_ok=True)

def calculate_hotspots(gdf, w):
    gdf = gdf.copy()
    g = G_Local(gdf["avg_summer_LST_Celsius"], w)
    gdf["Gi*"] = g.Zs
    return gdf

cities_gdfs = sorted(city_averaged_gdfs.keys())
cities_weights = set(loaded_city_knn_weights_averaged.keys())

print("\n📋 Städte in city_averaged_gdfs:", cities_gdfs)
print("📋 Städte in loaded_city_knn_weights_averaged:", sorted(cities_weights))

# nur Städte >= "Bayreuth" (alphabetisch) nehmen
start_city = "Passau"
start_index = cities_gdfs.index(start_city)
cities_to_process = cities_gdfs[start_index:]

print(f"\n🚀 Starte Berechnung ab: {start_city}")

for city in cities_to_process:
    if city not in loaded_city_knn_weights_averaged:
        print(f"⚠️  Kein Gewicht für {city} gefunden — überspringe.")
        continue

    print(f"\n✨ Bearbeite {city}...")

    gdf = city_averaged_gdfs[city]
    w = loaded_city_knn_weights_averaged[city]

    # Gi* berechnen
    result_gdf = calculate_hotspots(gdf, w)

    out_path = os.path.join(output_dir, f"{city.replace(' ', '_')}.geojson")

    result_gdf.to_file(out_path, driver="GeoJSON")
    print(f"✅ Ergebnis für {city} gespeichert: {out_path}")

    # Speicher freigeben
    del result_gdf



📋 Städte in city_averaged_gdfs: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']
📋 Städte in loaded_city_knn_weights_averaged: ['Aschaffenburg', 'Augsburg', 'Bamberg', 'Bayreuth', 'Erlangen', 'Fürth', 'Ingolstadt', 'Kempten (Allgäu)', 'Landshut', 'Munich', 'Nuremberg', 'Passau', 'Regensburg', 'Rosenheim', 'Schweinfurt', 'Würzburg']

🚀 Starte Berechnung ab: Passau

✨ Bearbeite Passau...


## Cold und Hot Spots klassifizieren

## nach Cold Spots mit p<0,05 filtern

# Gi* Analyse mit räumlichen Chunks
Apply spatial chunking to perform the Gi* analysis for each city in the `loaded_averaged_lst_gdf` GeoDataFrame using the pre-calculated KNN weights in the `city_knn_weights_averaged` dictionary. Explain the process and provide the code.

Gi* Analyse wurde mit räumlichen chunks durchgeführt für Speichereffizienz, vorher leider immer abgestürzt

## Define a spatial grid

### Subtask:
Create a spatial grid that covers the extent of each city's GeoDataFrame.


**Reasoning**:
The first step is to iterate through the city_averaged_gdfs, get the bounds of each city's geodataframe and create a spatial grid covering the extent.



In [ ]:
from shapely.geometry import box
import math

city_grids = {}
grid_cell_size = 1000 # Define grid cell size in meters (assuming projected CRS)

for city, gdf in city_averaged_gdfs.items():
    print(f"Creating grid for {city}...")
    if gdf.crs is None or gdf.crs.is_geographic:
        # Convert to a projected CRS for accurate grid creation
        try:
            print(f"  Converting {city} to EPSG:25832 for grid creation.")
            gdf_projected = gdf.to_crs(epsg=25832)
        except Exception as crs_e:
            print(f"  Error converting CRS for {city}: {crs_e}. Skipping grid for this city.")
            continue
    else:
        gdf_projected = gdf.copy()

    bounds = gdf_projected.total_bounds
    minx, miny, maxx, maxy = bounds

    # Calculate the number of rows and columns needed for the grid
    n_cols = int(math.ceil((maxx - minx) / grid_cell_size))
    n_rows = int(math.ceil((maxy - miny) / grid_cell_size))

    # Create the grid of polygons
    grid_cells = []
    for i in range(n_cols):
        for j in range(n_rows):
            cell_minx = minx + i * grid_cell_size
            cell_miny = miny + j * grid_cell_size
            cell_maxx = minx + (i + 1) * grid_cell_size
            cell_maxy = miny + (j + 1) * grid_cell_size
            grid_cells.append(box(cell_minx, cell_miny, cell_maxx, cell_maxy))

    grid_gdf = gpd.GeoDataFrame(geometry=grid_cells, crs=gdf_projected.crs)
    city_grids[city] = grid_gdf
    print(f"  Grid created for {city} with {len(grid_gdf)} cells.")

print("\nGrids created for all cities.")

Creating grid for Aschaffenburg...
  Converting Aschaffenburg to EPSG:25832 for grid creation.
  Grid created for Aschaffenburg with 144 cells.
Creating grid for Augsburg...
  Converting Augsburg to EPSG:25832 for grid creation.
  Grid created for Augsburg with 345 cells.
Creating grid for Bamberg...
  Converting Bamberg to EPSG:25832 for grid creation.
  Grid created for Bamberg with 100 cells.
Creating grid for Bayreuth...
  Converting Bayreuth to EPSG:25832 for grid creation.
  Grid created for Bayreuth with 150 cells.
Creating grid for Erlangen...
  Converting Erlangen to EPSG:25832 for grid creation.
  Grid created for Erlangen with 130 cells.
Creating grid for Fürth...
  Converting Fürth to EPSG:25832 for grid creation.
  Grid created for Fürth with 96 cells.
Creating grid for Ingolstadt...
  Converting Ingolstadt to EPSG:25832 for grid creation.
  Grid created for Ingolstadt with 304 cells.
Creating grid for Kempten (Allgäu)...
  Converting Kempten (Allgäu) to EPSG:25832 for gri

## Assign Data to Chunks

### Subtask:
Spatially join the city's GeoDataFrame with the grid to assign each data point (pixel) to a grid cell (chunk).

In [ ]:
# Dictionary to store city GeoDataFrames with assigned chunk IDs
city_data_with_chunks = {}

print("\nAssigning data points to spatial chunks...")

for city, gdf_city_averaged in city_averaged_gdfs.items():
    print(f"  Assigning data for {city} to chunks...")

    if city in city_grids:
        grid_gdf = city_grids[city]

        # Ensure both GeoDataFrames have the same CRS before spatial join
        if gdf_city_averaged.crs != grid_gdf.crs:
            print(f"    CRS mismatch for {city}. Converting city data to grid CRS ({grid_gdf.crs.to_epsg()}).")
            try:
                gdf_city_projected = gdf_city_averaged.to_crs(grid_gdf.crs)
            except Exception as crs_e:
                 print(f"    Error converting CRS for {city}: {crs_e}. Skipping chunk assignment for this city.")
                 continue
        else:
            gdf_city_projected = gdf_city_averaged.copy() # Use projected data if already in grid CRS

        # Perform a spatial join to assign a grid cell ID to each data point
        # Use 'within' predicate to assign points to the grid cell they fall within
        # We'll add a unique chunk ID to the grid first
        grid_gdf['chunk_id'] = range(len(grid_gdf))

        # Perform spatial join
        # Use 'left' join to keep all original data points, even if they fall outside the grid
        # (though ideally all points should be within the grid extent)
        # Suffixes are added to overlapping column names, we'll drop the grid geometry later
        gdf_city_with_chunks = gpd.sjoin(gdf_city_projected, grid_gdf[['geometry', 'chunk_id']], how="left", predicate="within")

        # Handle points outside the grid (chunk_id will be NaN) if any
        if gdf_city_with_chunks['chunk_id'].isnull().any():
            print(f"    Warning: Some points for {city} are outside the grid extent (NaN chunk_id).")
            # You might want to filter these out or handle them specifically
            # For now, we'll keep them but they won't be processed in chunks

        city_data_with_chunks[city] = gdf_city_with_chunks
        print(f"    Data for {city} assigned to chunks. Resulting GDF has {len(gdf_city_with_chunks)} rows.")

    else:
        print(f"  Grid not found for {city}. Skipping chunk assignment.")

print("\nChunk assignment completed for all cities.")

# The dictionary 'city_data_with_chunks' now contains GeoDataFrames for each city,
# with an added 'chunk_id' column indicating which grid cell each pixel belongs to.
# Example: city_data_with_chunks['Munich']


Assigning data points to spatial chunks...
  Assigning data for Aschaffenburg to chunks...
    CRS mismatch for Aschaffenburg. Converting city data to grid CRS (25832).
    Data for Aschaffenburg assigned to chunks. Resulting GDF has 69391 rows.
  Assigning data for Augsburg to chunks...
    CRS mismatch for Augsburg. Converting city data to grid CRS (25832).
    Data for Augsburg assigned to chunks. Resulting GDF has 164425 rows.
  Assigning data for Bamberg to chunks...
    CRS mismatch for Bamberg. Converting city data to grid CRS (25832).
    Data for Bamberg assigned to chunks. Resulting GDF has 121604 rows.
  Assigning data for Bayreuth to chunks...
    CRS mismatch for Bayreuth. Converting city data to grid CRS (25832).
    Data for Bayreuth assigned to chunks. Resulting GDF has 149925 rows.
  Assigning data for Erlangen to chunks...
    CRS mismatch for Erlangen. Converting city data to grid CRS (25832).
    Data for Erlangen assigned to chunks. Resulting GDF has 80705 rows.
 

## Process Each Chunk and Perform Gi* Analysis

### Subtask:
Iterate through each unique chunk for each city, filter the data for the chunk (with buffer), calculate KNN weights, perform Gi* analysis, and store results.

In [ ]:
# Dictionary to store Gi* results per city, combined from all chunks
city_gi_results_chunked = {}

# Define buffer distance (in meters, assuming projected CRS)
# This buffer is crucial to avoid edge effects in spatial analysis of chunks
# Points within the buffer but outside the chunk are used for neighbors, but not for the statistic itself.
buffer_distance = grid_cell_size * 1 # Example: buffer equal to one grid cell size

print(f"\nProcessing each chunk and performing Gi* analysis with a buffer of {buffer_distance} meters...")

for city, gdf_city_with_chunks in city_data_with_chunks.items():
    print(f"\n✨ Processing chunks for city: {city}...")

    if gdf_city_with_chunks.empty:
        print(f"  No data with chunk assignments for {city}. Skipping.")
        continue

    # Ensure we have the corresponding weight matrix available (if needed later, although chunking recalculates weights)
    # This part might need adjustment depending on whether you reuse city-level weights or calculate chunk-level weights
    # Given the plan, we calculate chunk-level weights, so the city-level matrix is not directly used here.
    # However, ensure the original data for the city is available if needed for context or re-joining.
    # We are using gdf_city_with_chunks which already has the necessary data.

    unique_chunks = gdf_city_with_chunks['chunk_id'].dropna().unique()
    print(f"  Found {len(unique_chunks)} chunks for {city}.")

    # List to store Gi* results for all chunks of the current city
    city_chunk_results = []

    # Iterate through each unique chunk ID
    for chunk_id in unique_chunks:
        print(f"    Processing chunk ID: {int(chunk_id)} for {city}...")

        # Get the geometry of the current chunk
        chunk_geometry = city_grids[city][city_grids[city]['chunk_id'] == chunk_id].geometry.iloc[0]

        # Create a buffered version of the chunk geometry
        buffered_chunk_geometry = chunk_geometry.buffer(buffer_distance)

        # Filter data points that are within the buffered chunk geometry
        # This includes points in the chunk AND points in the buffer zone
        gdf_chunk_buffered = gdf_city_with_chunks[
            gdf_city_with_chunks.within(buffered_chunk_geometry)
        ].copy() # Important: Copy to avoid SettingWithCopyWarning

        if gdf_chunk_buffered.empty:
            print(f"      No data points found within buffered chunk {int(chunk_id)}. Skipping.")
            continue

        # Filter data points that are within the original (unbuffered) chunk geometry
        # These are the points for which we will calculate and keep the Gi* results
        gdf_chunk_original = gdf_chunk_buffered[
            gdf_chunk_buffered.within(chunk_geometry)
        ].copy() # Important: Copy

        if gdf_chunk_original.empty:
             print(f"      No data points found within original chunk {int(chunk_id)}. Skipping Gi* calculation for this chunk.")
             continue

        # Ensure enough points for KNN in the buffered chunk
        min_points_for_knn = k_neighbors + 1 # Use the same k as before
        if len(gdf_chunk_buffered) >= min_points_for_knn:

            # Extract coordinates from the buffered chunk data for weight matrix calculation
            # The weight matrix must be built based on ALL points in the buffered area
            coords_buffered = np.array(list(zip(gdf_chunk_buffered.geometry.x, gdf_chunk_buffered.geometry.y)))

            # Extract the variable for analysis from the buffered chunk data
            # The variable values must also correspond to ALL points in the buffered area
            y_buffered = gdf_chunk_buffered['avg_summer_LST_Celsius'].values

            # Calculate KNN weights for the buffered chunk
            try:
                # Use the original indices from gdf_chunk_buffered to build the weight matrix
                # This is crucial so that the weight matrix indices match the data indices
                w_chunk_buffered = KNN.from_array(coords_buffered, k=k_neighbors, ids=gdf_chunk_buffered.index)
                w_chunk_buffered.transform = 'R' # Row standardize

                # Check for connectivity (optional but good practice)
                if w_chunk_buffered.n_components > 1:
                     print(f"      Warning: Weight matrix for buffered chunk {int(chunk_id)} is not fully connected ({w_chunk_buffered.n_components} components).")

                # Perform Gi* analysis on the buffered data using the buffered weight matrix
                # The results (Gi* values and p-values) will be calculated for all points in gdf_chunk_buffered
                gi_local_buffered = G_Local(y_buffered, w_chunk_buffered)

                # Add the Gi* results back to the buffered GeoDataFrame
                gdf_chunk_buffered['Gi_Star_buffered'] = gi_local_buffered.Gs
                gdf_chunk_buffered['Gi_Star_p_value_buffered'] = gi_local_buffered.p_sim

                # Now, extract the results ONLY for the points within the ORIGINAL chunk geometry
                # We need to join or map the results back to the original chunk data's index
                # The most reliable way is to merge based on the original index
                gi_results_for_original_chunk = gdf_chunk_buffered.loc[gdf_chunk_original.index, ['Gi_Star_buffered', 'Gi_Star_p_value_buffered']].copy()

                # Add these results as new columns to the original chunk GeoDataFrame
                gdf_chunk_original['Gi_Star'] = gi_results_for_original_chunk['Gi_Star_buffered']
                gdf_chunk_original['Gi_Star_p_value'] = gi_results_for_original_chunk['Gi_Star_p_value_buffered']


                # Determine significance for the original chunk points
                gdf_chunk_original['Gi_Star_sig'] = 'Nicht signifikant'
                gdf_chunk_original.loc[(gdf_chunk_original['Gi_Star_p_value'] < 0.05) & (gdf_chunk_original['Gi_Star'] < 0), 'Gi_Star_sig'] = 'Cold Spot (p<0.05)'
                gdf_chunk_original.loc[(gdf_chunk_original['Gi_Star_p_value'] < 0.05) & (gdf_chunk_original['Gi_Star'] > 0), 'Gi_Star_sig'] = 'Hot Spot (p<0.05)'
                # You can add more significance levels here

                # Append the results for the original chunk to the list
                city_chunk_results.append(gdf_chunk_original)

                print(f"      Gi* analysis successful for chunk {int(chunk_id)} ({len(gdf_chunk_original)} points in original chunk).")

            except Exception as gi_e:
                print(f"      Error during Gi* calculation for buffered chunk {int(chunk_id)}: {gi_e}. Skipping this chunk.")

        else:
            print(f"      Not enough data points ({len(gdf_chunk_buffered)}) in buffered chunk {int(chunk_id)} for KNN calculation (needs at least {min_points_for_knn}). Skipping.")


    # After processing all chunks for the city, concatenate the results
    if city_chunk_results:
        city_gi_results_chunked[city] = pd.concat(city_chunk_results)
        print(f"  Combined Gi* results for all chunks in {city}.")
        # Optional: Display head of combined results for the city
        # display(city_gi_results_chunked[city][['avg_summer_LST_Celsius', 'Gi_Star', 'Gi_Star_p_value', 'Gi_Star_sig']].head())
    else:
        print(f"  No Gi* results were generated for any chunk in {city}.")


print("\nGi* analysis with spatial chunking completed for all cities.")

# The dictionary 'city_gi_results_chunked' contains GeoDataFrames for each city
# with the Gi* results calculated using spatial chunking.
# Example: city_gi_results_chunked['Munich']


Processing each chunk and performing Gi* analysis with a buffer of 1000 meters...

✨ Processing chunks for city: Aschaffenburg...
  Found 90 chunks for Aschaffenburg.
    Processing chunk ID: 121 for Aschaffenburg...
      Gi* analysis successful for chunk 121 (319 points in original chunk).
    Processing chunk ID: 133 for Aschaffenburg...
      Gi* analysis successful for chunk 133 (76 points in original chunk).
    Processing chunk ID: 122 for Aschaffenburg...
      Gi* analysis successful for chunk 122 (784 points in original chunk).
    Processing chunk ID: 134 for Aschaffenburg...
      Gi* analysis successful for chunk 134 (58 points in original chunk).
    Processing chunk ID: 109 for Aschaffenburg...
      Gi* analysis successful for chunk 109 (634 points in original chunk).
    Processing chunk ID: 110 for Aschaffenburg...
      Gi* analysis successful for chunk 110 (1156 points in original chunk).
    Processing chunk ID: 98 for Aschaffenburg...
      Gi* analysis successfu

/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


      Gi* analysis successful for chunk 5 (645 points in original chunk).
    Processing chunk ID: 17 for Aschaffenburg...


/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


      Gi* analysis successful for chunk 17 (1030 points in original chunk).
    Processing chunk ID: 6 for Aschaffenburg...


...

      Gi* analysis successful for chunk 458 (357 points in original chunk).
    Processing chunk ID: 437 for Munich...
      Gi* analysis successful for chunk 437 (1545 points in original chunk).
    Processing chunk ID: 436 for Munich...
      Gi* analysis successful for chunk 436 (875 points in original chunk).
    Processing chunk ID: 457 for Munich...
      Gi* analysis successful for chunk 457 (66 points in original chunk).
    Processing chunk ID: 475 for Munich...
      Gi* analysis successful for chunk 475 (626 points in original chunk).
    Processing chunk ID: 517 for Munich...
      Gi* analysis successful for chunk 517 (589 points in original chunk).
    Processing chunk ID: 516 for Munich...
      Gi* analysis successful for chunk 516 (426 points in original chunk).
    Processing chunk ID: 496 for Munich...
      Gi* analysis successful for chunk 496 (841 points in original chunk).
    Processing chunk ID: 495 for Munich...
      Gi* analysis successful for chunk 495 (112

/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 4 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)
/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 3 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


      Gi* analysis successful for chunk 513 (484 points in original chunk).
    Processing chunk ID: 492 for Munich...
      Gi* analysis successful for chunk 492 (968 points in original chunk).
    Processing chunk ID: 493 for Munich...
      Gi* analysis successful for chunk 493 (694 points in original chunk).
    Processing chunk ID: 473 for Munich...
      Gi* analysis successful for chunk 473 (1144 points in original chunk).
    Processing chunk ID: 472 for Munich...
      Gi* analysis successful for chunk 472 (1110 points in original chunk).
    Processing chunk ID: 471 for Munich...
      Gi* analysis successful for chunk 471 (1100 points in original chunk).
    Processing chunk ID: 451 for Munich...
      Gi* analysis successful for chunk 451 (1148 points in original chunk).
    Processing chunk ID: 450 for Munich...
      Gi* analysis successful for chunk 450 (1128 points in original chunk).
    Processing chunk ID: 452 for Munich...
      Gi* analysis successful for chunk 452

/usr/local/lib/python3.11/dist-packages/libpysal/weights/distance.py:153: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  W.__init__(self, neighbors, id_order=ids, **kwargs)


      Gi* analysis successful for chunk 354 (349 points in original chunk).
    Processing chunk ID: 333 for Munich...
      Gi* analysis successful for chunk 333 (678 points in original chunk).
    Processing chunk ID: 396 for Munich...
      Gi* analysis successful for chunk 396 (1204 points in original chunk).
    Processing chunk ID: 375 for Munich...
      Gi* analysis successful for chunk 375 (6 points in original chunk).
    Processing chunk ID: 417 for Munich...
      Gi* analysis successful for chunk 417 (1731 points in original chunk).
    Processing chunk ID: 438 for Munich...
      Gi* analysis successful for chunk 438 (605 points in original chunk).
    Processing chunk ID: 312 for Munich...
      Gi* analysis successful for chunk 312 (543 points in original chunk).
    Processing chunk ID: 311 for Munich...
      Gi* analysis successful for chunk 311 (1156 points in original chunk).
    Processing chunk ID: 291 for Munich...
      Gi* analysis successful for chunk 291 (60

Variante mit chunks hat funktioniert, dauert aber ca 40 Minuten

In [ ]:
# Definieren Sie den Ordner in Google Drive, in dem die Ergebnisse gespeichert werden sollen
# Passen Sie den Pfad bei Bedarf an
output_folder_gi_results = "/content/drive/MyDrive/Cold Spots Bayern/gi_results_chunked/"
os.makedirs(output_folder_gi_results, exist_ok=True) # Erstellt den Ordner, falls er nicht existiert

print(f"Speichere Gi* Analyse-Ergebnisse (chunked) im Ordner: {output_folder_gi_results}")

# Schleife über das Dictionary der Gi* Ergebnisse pro Stadt
for city, gdf_gi_results in city_gi_results_chunked.items():
    if gdf_gi_results is not None and not gdf_gi_results.empty:
        # Erstellen Sie einen Dateinamen basierend auf dem Stadtnamen
        # Verwenden Sie ein Format, das GeoPandas schreiben kann, z.B. GeoJSON
        file_name = f"{city.replace(' ', '_')}_gi_results_chunked.geojson"
        file_path = os.path.join(output_folder_gi_results, file_name)

        try:
            # Speichern Sie den GeoDataFrame als GeoJSON-Datei
            gdf_gi_results.to_file(file_path, driver='GeoJSON')
            print(f"  Ergebnisse für {city} gespeichert unter: {file_path} ({len(gdf_gi_results)} Zeilen)")
        except Exception as e:
            print(f"  Fehler beim Speichern der Ergebnisse für {city}: {e}")
    else:
        print(f"  Keine Gi* Ergebnisse für {city} zum Speichern verfügbar.")

print("\nSpeichern der Gi* Analyse-Ergebnisse (chunked) abgeschlossen.")

Speichere Gi* Analyse-Ergebnisse (chunked) im Ordner: /content/drive/MyDrive/Cold Spots Bayern/gi_results_chunked/
  Ergebnisse für Aschaffenburg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/gi_results_chunked/Aschaffenburg_gi_results_chunked.geojson (69388 Zeilen)
  Ergebnisse für Augsburg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/gi_results_chunked/Augsburg_gi_results_chunked.geojson (164416 Zeilen)
  Ergebnisse für Bamberg gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/gi_results_chunked/Bamberg_gi_results_chunked.geojson (121600 Zeilen)
  Ergebnisse für Bayreuth gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/gi_results_chunked/Bayreuth_gi_results_chunked.geojson (149921 Zeilen)
  Ergebnisse für Erlangen gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/gi_results_chunked/Erlangen_gi_results_chunked.geojson (80701 Zeilen)
  Ergebnisse für Fürth gespeichert unter: /content/drive/MyDrive/Cold Spots Bayern/gi_results_

In [ ]:
display(city_gi_results_chunked['Munich'][['avg_summer_LST_Celsius', 'Gi_Star', 'Gi_Star_p_value', 'Gi_Star_sig']].head())

,avg_summer_LST_Celsius,Gi_Star,Gi_Star_p_value,Gi_Star_sig
1010628,34.020015,0.001246,0.135,Nicht signifikant
1013385,32.094435,0.001203,0.003,Hot Spot (p<0.05)
1013386,32.066427,0.001203,0.003,Hot Spot (p<0.05)
1013387,31.919926,0.001203,0.003,Hot Spot (p<0.05)
1013388,31.930275,0.001203,0.003,Hot Spot (p<0.05)


# Weiterverarbeitung der Cold-Spot-Ergebnisse

Schritte und weitere Ziele:
- gdfs filtern nach ausschließlich Cold Spots jeder Stadt
- OSMnx Erreichbarkeitsanalysen
- Karte für jede Stadt, die Cold Spots genau und geometriegenau eingezeichnet hat, und die Einzugsgebiet eingezeichnet hat (vgl. cooler Stadtplan Münster https://geo.stadt-muenster.de/coolerstadtplan/)